# Week 7, Day 3: Functions as values

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w007d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def double(n):
    return n * 2

print(double(5))
print(type(double))
print(double)

In [ ]:
def shout(text):
    return text.upper() + "!"

yell = shout
print(yell("hello"))
print(yell is shout)

> **Key idea.** `f` is the function itself. `f()` **calls** it and gives you its result. Passing a function around means writing its name **without** brackets.

In [ ]:
def apply_twice(f, x):
    return f(f(x))

def add_three(n):
    return n + 3

def double(n):
    return n * 2

print(apply_twice(add_three, 10))
print(apply_twice(double, 10))

In [ ]:
def transform(text, change):
    result = ""
    for ch in text:
        result += change(ch)
    return result

def double_letter(ch):
    return ch * 2

def hide_vowel(ch):
    return "*" if ch in "aeiou" else ch

print(transform("hello", double_letter))
print(transform("hello", hide_vowel))
print(transform("hello", str.upper))

In [ ]:
def count_if(test, text):
    count = 0
    for ch in text:
        if test(ch):
            count += 1
    return count

print(count_if(str.isdigit, "R2-D2 and C-3PO"))
print(count_if(str.isupper, "R2-D2 and C-3PO"))

In [ ]:
print(max("fig", "banana", "kiwi"))
print(max("fig", "banana", "kiwi", key=len))
print(min("Bob", "alice", "Carol"))
print(min("Bob", "alice", "Carol", key=str.lower))

In [ ]:
words = "Banana apple Cherry date".split()
print(sorted(words))
print(sorted(words, key=str.lower))
print(sorted(words, key=len))

In [ ]:
square = lambda n: n * n
print(square(7))

words = "fig banana kiwi apple".split()
print(max(words, key=lambda w: w[-1]))
print(sorted(words, key=lambda w: (len(w), w)))

> **Tip.** Use `lambda` for short functions passed straight into another function, like a `key`. If you'd give it a name, or it needs more than one simple expression, write a `def` with a docstring instead.

In [ ]:
def make_multiplier(factor):
    def multiply(n):
        return n * factor
    return multiply

double = make_multiplier(2)
triple = make_multiplier(3)
print(double(10), triple(10))

In [ ]:
def make_money_formatter(symbol, places=2):
    def fmt(amount):
        return f"{symbol}{amount:,.{places}f}"
    return fmt

naira = make_money_formatter("₦")
dollars = make_money_formatter("$", 0)
print(naira(1234.5), dollars(99.4))

### Exercise: Apply n times

Write `apply_n(f, x, n)` that applies `f` to `x`, `n` times in a row, and returns the result. `apply_n(f, x, 0)` returns `x` unchanged.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several calls\"\n    assert apply_n(lambda v: v + 1, 0, 5) == 5\n    assert apply_n(lambda v: v * 2, 1, 10) == 1024\n    assert apply_n(str.upper, \"hi\", 0) == \"hi\"")

### Exercise: Transform each character

Write `transform(text, change)` that returns a new string where every character `ch` has been replaced by `change(ch)`. Then write `swap(ch)`, which swaps capitals and lower case and leaves other characters alone, so that `transform("Hello, World", swap)` gives `"hELLO, wORLD"`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_transform():\n    \"transform works with any function\"\n    assert transform(\"abc\", lambda c: c * 2) == \"aabbcc\"\n    assert transform(\"\", str.upper) == \"\"\ndef test_swap():\n    \"swap\"\n    assert transform(\"Hello, World\", swap) == \"hELLO, wORLD\"")

### Exercise: Count if

Write `count_if(test, text)` that returns how many characters of `text` make `test` return `True`. Then use it with a `lambda` to print how many characters in `"Python Atlas 2026!"` are **not** letters.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_function():\n    \"count_if\"\n    assert count_if(str.isdigit, \"a1b22\") == 3\n    assert count_if(lambda c: c == \"s\", \"mississippi\") == 4\ndef test_output():\n    \"the printed count\"\n    expect_output(\"7\")\n    assert \"lambda\" in source, \"Use a lambda for the test.\"")

### Exercise: Choosing with key

Ask for some words on one line with the prompt `Words: `. Print three lines, using `max` or `min` with a `key` each time: `Longest: ` (the first if tied), `Shortest: `, and `Most vowels: ` (the first if tied).

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_words():\n    \"several word lists\"\n    expect_output(\"Words: fig banana kiwi education\\nLongest: education\\nShortest: fig\\nMost vowels: education\", inputs=[\"fig banana kiwi education\"], echo=True)\n    expect_output(\"Words: aa bb cc\\nLongest: aa\\nShortest: aa\\nMost vowels: aa\", inputs=[\"aa bb cc\"], echo=True)\ndef test_key():\n    \"uses key=\"\n    assert source.count(\"key=\") >= 3, \"Use key= with max or min for each line.\"")

### Exercise: Make an adder

Write `make_adder(n)` that returns a function which adds `n` to whatever it's given. `add5 = make_adder(5)` then `add5(10)` is 15.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several adders\"\n    add5 = make_adder(5)\n    minus2 = make_adder(-2)\n    assert add5(10) == 15 and add5(0) == 5\n    assert minus2(10) == 8\n    assert make_adder(1)(make_adder(1)(0)) == 2")

### Exercise: Compose

Write `compose(f, g)` that returns a new function which, given `x`, returns `f(g(x))`. Then build `shout_trimmed = compose(str.upper, str.strip)` and print `shout_trimmed("  hello  ")`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_compose():\n    \"compose\"\n    inc = lambda x: x + 1\n    dbl = lambda x: x * 2\n    assert compose(inc, dbl)(5) == 11, \"compose(f, g)(x) is f(g(x)): g first.\"\n    assert compose(dbl, inc)(5) == 12\ndef test_output():\n    \"prints HELLO\"\n    expect_output(\"HELLO\")")

### Exercise: Pick an operation

Write `get_operation(symbol)` that returns a **function** for `+`, `-`, `*` or `/`, each taking two numbers. Use lambdas. For any other symbol it returns `None`. Then the program asks `a: `, `op: ` and `b: ` and prints the result, or `Unknown operation`.

In [ ]:
def get_operation(symbol):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_function():\n    \"get_operation returns functions\"\n    assert get_operation(\"+\")(2, 3) == 5 and get_operation(\"-\")(2, 3) == -1\n    assert get_operation(\"*\")(2, 3) == 6 and get_operation(\"/\")(3, 2) == 1.5\n    assert get_operation(\"%\") is None\ndef test_program():\n    \"the program\"\n    expect_output(\"a: 6\\nop: *\\nb: 7\\n42.0\", inputs=[\"6\", \"*\", \"7\"], echo=True)\n    expect_output(\"a: 6\\nop: ^\\nb: 7\\nUnknown operation\", inputs=[\"6\", \"^\", \"7\"], echo=True)")

### Bug hunt: Called too soon

This crashes with `TypeError: apply_twice() missing ...` or `'str' object is not callable`. It should print `HELLO!!`. Fix the call.

In [ ]:
def apply_twice(f, x):
    return f(f(x))

def excite(text):
    return text.upper() + "!"

print(apply_twice(excite(), "hello"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"prints HELLO!!\"\n    expect_output(\"HELLO!!\")")

### Bug hunt: Comparing the wrong thing

This should print the word with the **most letters**, but it prints `pear`. Fix it.

In [ ]:
words = "banana fig pear".split()
print(max(words))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"prints banana\"\n    expect_output(\"banana\")")

### Bug hunt: A lambda that returns nothing useful

`by_last_letter` should sort words by their last letter, but this crashes with a `SyntaxError`. A lambda can only hold an expression. Fix it.

In [ ]:
words = "kiwi apple fig".split()
print(sorted(words, key=lambda w: return w[-1]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"sorted by last letter\"\n    expect_output(\"['apple', 'fig', 'kiwi']\")")

### Puzzle: Make a multiplier

Arrange the lines into a function that makes multiplier functions, then use it. One line is a trap.

(Put these lines in order on the website.)

### Exercise: What do they print?

Work these out without running them.

```python
def make(n):
    return lambda x: x * n + 1

f = make(2)
g = make(10)
print(f(3))           # A
print(g(f(0)))        # B
print(max("ab", "c", "def", key=len))   # C
```

In [ ]:
answer_a = ___
answer_b = ___
answer_c = "___"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each line\"\n    assert answer_a == 7, \"f is x * 2 + 1.\"\n    assert answer_b == 11, \"f(0) is 1, then g(1) is 1 * 10 + 1.\"\n    assert answer_c == \"def\", \"The longest by len.\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: Newton's method for any function

In Week 4 you used Newton's method to find square roots. It works for almost any function: to find where `f(x) = 0`, start from a guess `x` and repeat `x = x - f(x) / slope`, where `slope` is how steeply `f` is rising at `x`.

1. Write `derivative(f, h=1e-6)` that **returns a function** giving the approximate slope of `f` at any `x`: `(f(x + h) - f(x - h)) / (2 * h)`.
2. Write `newton(f, x, tolerance=1e-10)` that repeats the Newton step, using `derivative(f)`, until `abs(f(x)) < tolerance`, and returns `x`. Stop after 100 steps if it hasn't got there.
3. Print the positive root of `x² − 2` (that's √2) and of `x³ − x − 1`, each rounded to 6 places, starting from 1.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_derivative():\n    \"derivative returns a slope function\"\n    d = derivative(lambda x: x * x)\n    assert abs(d(3) - 6) < 1e-4, \"The slope of x*x at 3 is 6.\"\n    d = derivative(lambda x: x ** 3)\n    assert abs(d(2) - 12) < 1e-4\ndef test_newton():\n    \"newton finds roots\"\n    assert abs(newton(lambda x: x * x - 2, 1) - 2 ** 0.5) < 1e-8\n    assert abs(newton(lambda x: x * x - 9, 1) - 3) < 1e-8\ndef test_output():\n    \"prints both roots\"\n    expect_output(\"1.414214\\n1.324718\")")

## Warm-up

The last two lines show `double` itself, not a call to it. It has a type, `function`, and Python can print it. That's because a function is an object, like any other value.

## Functions are values

Since a function is a value, you can do with it the things you do with values. You can give it another name:

`yell = shout` doesn't call anything (there are no brackets). It ties a second label, `yell`, to the same function object. Press X-ray: both names point at one function.

## Passing functions to functions

Because a function is a value, you can pass it as an argument. The receiving function can then call it:

`apply_twice` doesn't know or care what `f` does. It just knows how to call it twice. A function that takes or returns other functions is called a **higher-order function**, and it lets one piece of code handle many jobs.

### One loop, many jobs

Last week you wrote several loops that went through a string, changing each character. They differed in only one line: what to do to a character. Pass that in as a function:

The last line passes `str.upper`, the method itself, as the function. Every string method can be used this way.

## key=: choosing what to compare

`max` and `min` take an optional `key` function. Instead of comparing the values themselves, they compare `key(value)`:

Plain `max` compares strings alphabetically, so `"kiwi"` wins. With `key=len`, it compares lengths, so `"banana"` wins, but it still gives you back the word, not the length. `key=str.lower` compares the lower-case versions, so capitals no longer jump the queue.

`sorted` takes a `key` too. It returns a **list** (next week's topic) of the items in order:

## lambda: a function on the spot

Writing a whole `def` for a one-line function you'll use once is clumsy. A **lambda** makes a small function inside an expression:

`lambda n: n * n` means "a function that takes `n` and returns `n * n`". The part after the colon must be a single **expression**: no `if` statements, loops or assignments, though a conditional expression is fine. Its value is returned automatically, with no `return` keyword.

The middle line finds the word whose **last letter** is greatest. The last line sorts by length, and for equal lengths alphabetically, by giving each word a pair of values to compare.

## Functions that make functions

A function can define another function inside itself and return it:

Each call to `make_multiplier` builds a **new** function. Here's the remarkable part: `multiply` uses `factor`, which is a local variable of `make_multiplier`, and that call has already finished. Yet `double` still knows its factor is 2.

The inner function **remembers** the variables of the function it was made in. A function that carries remembered variables like this is called a **closure**. It's the **E** in LEGB from last week: when `multiply` looks up `factor`, it isn't local, so Python looks in the **enclosing** function's variables, which were kept alive for it. Step through it in X-ray and watch the frames.

You configure the formatter once, then use it many times without repeating the settings.

## Practice

## Quiz

## Daily challenge

## Before you go

Explain in your own words how `double = make_multiplier(2)` still knows its factor after `make_multiplier` has finished. Then write a `key` function that sorts words by their number of vowels. Tomorrow: top-down design, which means starting a program from the big picture and filling in the functions it needs, step by step.